# F1 Championship Prediction — Exploratory Data Analysis
End-to-end walkthrough: data loading → EDA → feature analysis → model training → prediction evaluation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'SRC'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)

RAW  = os.path.join('..', 'Data', 'raw data')
PROC = os.path.join('..', 'Data', 'processed')

## 1. Load Raw Data

In [ ]:
races        = pd.read_csv(f'{RAW}/races.csv')
results      = pd.read_csv(f'{RAW}/results.csv')
drivers      = pd.read_csv(f'{RAW}/drivers.csv')
constructors = pd.read_csv(f'{RAW}/constructors.csv')
status       = pd.read_csv(f'{RAW}/status.csv')
driver_standings = pd.read_csv(f'{RAW}/driver_standings.csv')

print('races:       ', races.shape)
print('results:     ', results.shape)
print('drivers:     ', drivers.shape)
print('status:      ', status.shape)
results.head()

## 2. Seasons Overview

In [ ]:
races_per_year = races.groupby('year')['raceId'].count()
races_per_year.plot(kind='bar', title='Races per Season', color='steelblue')
plt.xlabel('Year')
plt.ylabel('Number of Races')
plt.tight_layout()
plt.show()
print(f'Dataset covers {races["year"].min()} to {races["year"].max()} — {races["year"].nunique()} seasons')

## 3. DNF Analysis

In [ ]:
finished_ids = status.loc[status['status'] == 'Finished', 'statusId'].values
results['dnf'] = (~results['statusId'].isin(finished_ids)).astype(int)

df = results.merge(races[['raceId','year']], on='raceId', how='left')
dnf_by_year = df.groupby('year')['dnf'].mean() * 100

dnf_by_year.plot(title='Average DNF Rate per Season (%)', color='crimson')
plt.ylabel('DNF Rate (%)')
plt.xlabel('Year')
plt.tight_layout()
plt.show()

## 4. Points Distribution — Modern Era (2010+)

In [ ]:
modern = df[df['year'] >= 2010].copy()
modern['points'] = pd.to_numeric(modern['points'], errors='coerce').fillna(0)

season_pts = modern.groupby(['year','driverId'])['points'].sum().reset_index()

sns.boxplot(data=season_pts, x='year', y='points')
plt.xticks(rotation=90)
plt.title('Distribution of Driver Season Points (2010–2024)')
plt.tight_layout()
plt.show()

## 5. Top 10 Most Successful Drivers (by wins)

In [ ]:
results['positionOrder'] = pd.to_numeric(results['positionOrder'], errors='coerce')
wins = results[results['positionOrder'] == 1].groupby('driverId').size().reset_index(name='wins')
wins = wins.merge(drivers[['driverId','forename','surname']], on='driverId')
wins['driver'] = wins['forename'] + ' ' + wins['surname']
wins = wins.nlargest(10, 'wins')

sns.barplot(data=wins, x='wins', y='driver', palette='viridis')
plt.title('Top 10 Drivers by Race Wins (All Time)')
plt.xlabel('Wins')
plt.ylabel('')
plt.tight_layout()
plt.show()

## 6. Feature Correlation Heatmap

In [ ]:
features = pd.read_csv(f'{PROC}/features.csv')
print(f'Features shape: {features.shape}')

cols_of_interest = [
    'avg_finish_pos', 'points_sum', 'win_rate', 'podium_rate',
    'dnf_rate', 'points_per_race', 'quali_to_race_delta',
    'team_final_points', 'prev_season_points', 'champ_position'
]
corr = features[cols_of_interest].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 7. Train Models

In [ ]:
from model import train_model
reg_model, clf_model = train_model()

## 8. Predict & Evaluate — 2023 Season

In [ ]:
from predict import predict_championship
results_2023 = predict_championship(2023)

## 9. Predicted vs Actual — Scatter Plot

In [ ]:
valid = results_2023.dropna(subset=['Actual Position'])

plt.figure(figsize=(8, 8))
plt.scatter(valid['Actual Position'], valid['Predicted Score'], color='steelblue', s=80)

for _, row in valid.iterrows():
    plt.annotate(row['Driver'].split()[-1],
                 (row['Actual Position'], row['Predicted Score']),
                 textcoords='offset points', xytext=(6, 4), fontsize=8)

# Perfect prediction line
lims = [1, int(valid['Actual Position'].max()) + 1]
plt.plot(lims, lims, 'r--', alpha=0.5, label='Perfect prediction')

plt.xlabel('Actual Championship Position')
plt.ylabel('Predicted Score (lower = better)')
plt.title('2023 F1 Championship — Predicted vs Actual')
plt.legend()
plt.tight_layout()
plt.show()

## 10. Feature Importance

In [ ]:
FEATURE_COLUMNS = [
    'avg_finish_pos','std_finish_pos','races_started','points_sum',
    'avg_grid_pos','win_rate','podium_rate','dnf_rate','points_per_race',
    'quali_to_race_delta','sprint_points_sum','team_final_points',
    'team_final_position','prev_season_points','prev_season_avg_pos','prev_season_win_rate'
]
imp = pd.Series(reg_model.feature_importances_, index=FEATURE_COLUMNS).sort_values()

imp.plot(kind='barh', title='Feature Importances — Random Forest', color='steelblue')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()